#  Evaluation, Comparison, Error Analysis & Demo
### CSC60904 Deep Learning — Group Assignment

Picks up where (Model 1: `BaselineCNN`) and (Model 2: ResNet18 transfer learning) left off. Re-loads their saved checkpoints, evaluates both on the **same** held-out test set, and produces:

1. Accuracy / Precision / Recall / F1 (per-class + macro) for both models
2. Confusion matrix for both models
3. A side-by-side comparison table + bar chart (feeds Report §6)
4. Qualitative error analysis: most-confident **wrong** predictions per class per model (feeds Report §6 Error Analysis)
5. An interactive Gradio demo (feeds Task 3 "lightweight demo interface" requirement)

> Run this **after** Member 2's and Member 3's cells have already executed in this runtime (or after reloading their saved `.pth` checkpoints in a fresh runtime — see the note in Section 0).

## 0. Setup

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("results", exist_ok=True)
os.makedirs("results/errors", exist_ok=True)

**If starting in a fresh runtime** (no `train_loader`/`test_loader`/`class_names` already in memory from earlier cells), rebuild the test loader exactly as Members 2/3 did:

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("./data/processed")
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

eval_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

test_dataset = datasets.ImageFolder(str(PROCESSED_DIR / "test"), transform=eval_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Use the class list Member 3 saved, so ordering is guaranteed consistent
# with the checkpoint's output layer.
with open("results/model2_class_names.json") as f:
    class_names = json.load(f)
num_classes = len(class_names)
print("Classes (fixed order from Member 3's run):", class_names)

## 1. Re-declare both model architectures
Must match Members 2 and 3's definitions **exactly**, or `state_dict` loading will fail or silently mismatch.

In [ ]:
class BaselineCNN(nn.Module):
    """Model 1 — Member 2's architecture, unchanged."""
    def __init__(self, num_classes=3):
        super(BaselineCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def build_model2(num_classes, freeze_backbone=False):
    """Model 2 — Member 3's ResNet18 + custom head, unchanged."""
    model = models.resnet18(weights=None)  # weights loaded from checkpoint below
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model.to(device)

## 2. Load both checkpoints

⚠️ **Update these two paths** to match whatever Member 2 / Member 3 actually saved.
Member 2's save call was `torch.save(model.state_dict(), "model1_baseline_cnn.pth")`.
Member 3's `train_model()` saved the best checkpoint to a `ckpt` path passed into the function — check that cell for the exact filename and update `MODEL2_CKPT` below.

In [ ]:
MODEL1_CKPT = "model1_baseline_cnn.pth"
MODEL2_CKPT = "models/best_model2.pt"

model1 = BaselineCNN(num_classes=num_classes).to(device)
model1.load_state_dict(torch.load(MODEL1_CKPT, map_location=device))
model1.eval()

model2 = build_model2(num_classes=num_classes)
model2.load_state_dict(torch.load(MODEL2_CKPT, map_location=device))
model2.eval()

print("Both checkpoints loaded successfully.")

## 3. Run both models over the test set

In [ ]:
def get_predictions(model, loader, dataset):
    all_preds, all_labels, all_confs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            confs, preds = torch.max(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_confs.extend(confs.cpu().numpy())
    file_paths = [s[0] for s in dataset.samples]  # ImageFolder keeps order
    return np.array(all_labels), np.array(all_preds), np.array(all_confs), file_paths


y_true, pred1, conf1, file_paths = get_predictions(model1, test_loader, test_dataset)
_,      pred2, conf2, _          = get_predictions(model2, test_loader, test_dataset)

## 4. Classification reports (Accuracy / Precision / Recall / F1)

In [ ]:
def save_report(y_true, y_pred, name):
    report = classification_report(y_true, y_pred, target_names=class_names,
                                     output_dict=True, zero_division=0)
    print(f"\n===== {name} — Classification Report =====")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    with open(f"results/{name}_classification_report.json", "w") as f:
        json.dump(report, f, indent=2)
    return report

report1 = save_report(y_true, pred1, "model1_baseline")
report2 = save_report(y_true, pred2, "model2_resnet18")

## 5. Confusion matrices (side-by-side figure for the report)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, y_pred, title in [(axes[0], pred1, "Model 1: Baseline CNN"),
                           (axes[1], pred2, "Model 2: ResNet18 Transfer")]:
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
    ax.set_title(title)
fig.tight_layout()
fig.savefig("results/confusion_matrices_comparison.png", dpi=150)
plt.show()
print("Saved results/confusion_matrices_comparison.png")

## 6. Model comparison table + bar chart

Includes **fire-class recall** specifically — the highest-stakes metric, since a missed fire is the costliest error type (see Report §7 Ethics).

In [ ]:
def summarize(report, y_true, y_pred):
    fire_idx = class_names.index("fire") if "fire" in class_names else None
    fire_recall = report[class_names[fire_idx]]["recall"] if fire_idx is not None else None
    return {
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "fire_class_recall": fire_recall,
    }

summary1 = summarize(report1, y_true, pred1)
summary2 = summarize(report2, y_true, pred2)

comparison = {"Model 1 (Baseline CNN)": summary1, "Model 2 (ResNet18 Transfer)": summary2}
with open("results/model_comparison_summary.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("\n===== Model Comparison Summary =====")
print(f"{'Metric':<20}{'Model 1':>15}{'Model 2':>15}")
for k in summary1:
    v1, v2 = summary1[k], summary2[k]
    v1s = f"{v1:.4f}" if v1 is not None else "n/a"
    v2s = f"{v2:.4f}" if v2 is not None else "n/a"
    print(f"{k:<20}{v1s:>15}{v2s:>15}")

In [ ]:
metrics_to_plot = ["accuracy", "macro_f1", "fire_class_recall"]
x = np.arange(len(metrics_to_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width/2, [summary1[m] for m in metrics_to_plot], width, label="Model 1: Baseline CNN")
ax.bar(x + width/2, [summary2[m] for m in metrics_to_plot], width, label="Model 2: ResNet18 Transfer")
ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Macro F1", "Fire-class Recall"])
ax.set_ylim(0, 1.05)
ax.set_title("Model 1 vs Model 2 — Key Metrics")
ax.legend()
for i, m in enumerate(metrics_to_plot):
    ax.text(i - width/2, summary1[m] + 0.01, f"{summary1[m]:.3f}", ha="center", fontsize=9)
    ax.text(i + width/2, summary2[m] + 0.01, f"{summary2[m]:.3f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig("results/model_comparison_bar.png", dpi=150)
plt.show()
print("Saved results/model_comparison_bar.png")

## 7. Error analysis

Top-5 most-confident **wrong** predictions per class, per model. These go directly into Report §6 as figures with captions.

In [ ]:
def error_analysis(y_true, y_pred, confs, file_paths, model_tag):
    out_root = f"results/errors/{model_tag}"
    os.makedirs(out_root, exist_ok=True)
    wrong_idx = np.where(y_true != y_pred)[0]
    print(f"\n{model_tag}: {len(wrong_idx)} / {len(y_true)} test images misclassified "
          f"({len(wrong_idx)/len(y_true):.1%})")

    for true_cls in range(num_classes):
        cls_wrong = [i for i in wrong_idx if y_true[i] == true_cls]
        cls_wrong_sorted = sorted(cls_wrong, key=lambda i: -confs[i])[:5]
        if not cls_wrong_sorted:
            continue

        fig, axes = plt.subplots(1, len(cls_wrong_sorted), figsize=(3 * len(cls_wrong_sorted), 3.2))
        if len(cls_wrong_sorted) == 1:
            axes = [axes]
        for ax, i in zip(axes, cls_wrong_sorted):
            from PIL import Image as PILImage
            img = PILImage.open(file_paths[i]).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"true: {class_names[true_cls]}\npred: {class_names[y_pred[i]]} "
                         f"({confs[i]:.0%})", fontsize=9)
            ax.axis("off")
        fig.suptitle(f"{model_tag} — most-confident mistakes on true '{class_names[true_cls]}'",
                     fontsize=10)
        fig.tight_layout()
        fig.savefig(f"{out_root}/errors_true_{class_names[true_cls]}.png", dpi=130)
        plt.show()

    return len(wrong_idx)

n_wrong_1 = error_analysis(y_true, pred1, conf1, file_paths, "model1_baseline")
n_wrong_2 = error_analysis(y_true, pred2, conf2, file_paths, "model2_resnet18")

print("\nAll evaluation artifacts saved under results/ — commit these to the "
      "GitHub repo and pull them into Report §6 (Experiments & Results) and "
      "the Error Analysis subsection.")

---
## 8. Interactive Demo (Gradio)

Gradio is used instead of Streamlit because it runs directly inline in Colab with zero extra setup (Streamlit needs ngrok/localtunnel in Colab, which adds friction for a live demo during Task 2/Task 5). If your team prefers a standalone Streamlit app instead, use the separate `app.py` — just run `streamlit run app.py` locally.

Run this cell **after** Section 2 above (so `model2`, `class_names`, and `device` already exist).

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
from PIL import Image

demo_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

# Uses Model 2 (ResNet18 transfer) for the live demo since it's the
# stronger of the two (see results/model_comparison_summary.json).
DEMO_MODEL = model2
DEMO_MODEL.eval()


def predict(image: Image.Image):
    if image is None:
        return {}
    img = image.convert("RGB")
    tensor = demo_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = DEMO_MODEL(tensor)
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}


with gr.Blocks(title="FireEye-Nepal Demo") as demo:
    gr.Markdown(
        "# FireEye-Nepal: Forest Fire & Smoke Detector\n"
        "Upload a photo from a forest camera, drone, or phone. The model "
        "(ResNet18, transfer-learned) predicts whether it shows **fire**, "
        "**smoke**, or **no fire** — a prototype early-warning building "
        "block for CSC60904's group assignment."
    )
    with gr.Row():
        img_input = gr.Image(type="pil", label="Upload image")
        label_output = gr.Label(num_top_classes=3, label="Prediction")
    img_input.change(fn=predict, inputs=img_input, outputs=label_output)

    gr.Markdown(
        "*Prototype for CSC60904 Deep Learning group assignment. Not a "
        "validated operational fire-detection system — trained on a "
        "global/non-Nepal-specific dataset. See report §7 (Ethics) and "
        "§8 (Deployment Considerations) for limitations.*"
    )

# share=True gives a public URL you can click during the Task 5 live demo —
# no ngrok account needed.
demo.launch(share=True, debug=True)

---
## Next steps for Member 4
See `MEMBER4_CHECKLIST.md` for: repo folder structure, a pre-filled README template, and the final code-integration checklist (fresh-runtime test, checkpoint filenames, commit history check).